# Vacuum Reference Run

Baseline run **without** a plasma neutralizer.
Establishes unneutralized beam propagation and verifies K_eff/K0 ≈ 1.

**Beamline**: `buncher → [plasma neutralizer] → solenoid → Q1 → Q2 → spiral inflector`

> Tip: set `MAX_STEPS = 500` and `DIAG_PERIOD = 100` for a smoke test first.


In [ ]:
from pathlib import Path
import os, sys, subprocess, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate project root
_ROOT = Path.cwd()
while _ROOT.name != 'plasma_column' and _ROOT.parent != _ROOT:
    _ROOT = _ROOT.parent
if str(_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_ROOT / 'src'))

WORK           = Path.home() / 'Work' / 'simulation_codes-working'
WARPX_DATA_DIR = WORK / 'warpx-data'
RESULTS_DIR    = _ROOT / 'results'  # Processed diagnostic CSV/JSON results
RUNS_DIR       = _ROOT / 'runs'     # Raw simulation output directories (gitignored)
PLOTS_DIR      = _ROOT / 'plots'    # Generated publication figures
RESULTS_DIR.mkdir(exist_ok=True)
RUNS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

os.environ['WARPX_DATA_DIR']  = str(WARPX_DATA_DIR)
os.environ['LD_LIBRARY_PATH'] = (
    str(WORK / 'warpx' / 'install' / 'lib') + ':'
    + os.environ.get('LD_LIBRARY_PATH', '')
)
print('Python :', sys.executable)
print('ROOT   :', _ROOT)
print('WarpX data:', WARPX_DATA_DIR)


In [ ]:
from plasma_column.notebook_utils import print_simulation_config
SCRIPT = _ROOT / 'scripts' / 'plasma_column_mcc_picmi_v7.py'
_DEFAULTS = {
    'script':               'scripts/plasma_column_mcc_picmi_v7.py',
    'case':                 'vacuum_reference',
    'gas':                  'H2',
    'neutralization':       0.0,
    'mcc':                  'none',
    'pressure_torr':        '1e-5',
    'plasma_age [s]':       '2e-4',
    'max_steps':            120000,
    'diag_period':          5000,
    'reduced_diag_period':  100,
    'nx / ny / nz':         '32 / 32 / 256',
}
_OVERRIDES = {}  # e.g. {'max_steps': 500, 'diag_period': 100}
print_simulation_config(
    notebook_title='Vacuum Reference Run',
    defaults=_DEFAULTS, overrides=_OVERRIDES,
    extra_info={'output root': str(RUNS_DIR / 'vacuum_reference')},
)


## 1. Configure


In [ ]:
MAX_STEPS      = 120000   # reduce to 500 for smoke test
DIAG_PERIOD    = 5000
REDUCED_PERIOD = 100
NX, NY, NZ     = 32, 32, 256
OUT_DIR = RUNS_DIR / 'vacuum_reference'
OUT_DIR.mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable, str(SCRIPT), '--run',
    '--output_dir',          str(OUT_DIR),
    '--gas',                 'H2',
    '--neutralization',      '0.0',
    '--mcc',                 'none',
    '--pressure_torr',       '1e-5',
    '--plasma_age',          '2e-4',
    '--max_steps',           str(MAX_STEPS),
    '--diag_period',         str(DIAG_PERIOD),
    '--reduced_diag_period', str(REDUCED_PERIOD),
    '--reduced_diag_dir',    'reducedfiles/',
    '--nx', str(NX), '--ny', str(NY), '--nz', str(NZ),
    '--warpx_data_dir',      str(WARPX_DATA_DIR),
]
print('Command:', ' '.join(cmd))


## 2. Run

Uncomment `subprocess.run` to start the simulation.


In [ ]:
# result = subprocess.run(cmd, check=True)
# print('Exit code:', result.returncode)
print('Ready — uncomment subprocess.run to launch.')


## 3. Load diagnostics


In [ ]:
from plasma_column.plotting import (
    setup_publication_style,
    plot_multi_case_neutralization,
    plot_neutralization_evolution,
    plot_particle_counts,
    plot_keff_over_k0,
    plot_species_growth_rates,
    plot_neutralization_panel,
    plot_bunched_beam_keff,
    plot_keff_pressure_scan,
    plot_radial_density_profile,
    plot_neutralization_vs_z,
    plot_phase_space,
    save_figure,
)
from plasma_column.diagnostics import (
    load_particle_number_diagnostic,
    compute_particle_number_metrics,
    DataLoader,
)
import warnings
setup_publication_style()
print('Plotting helpers loaded.')


In [ ]:
import warnings
_OUT = RUNS_DIR / 'vacuum_reference'
_diag_candidates = [
    _OUT / 'reducedfiles' / 'ParticleNumber_red.txt',
    _OUT / 'neutralization_from_particle_number.csv',
]
_diag = next((p for p in _diag_candidates if p.exists()), None)
if _diag:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        df_vac = load_particle_number_diagnostic(_diag)
        df_vac = compute_particle_number_metrics(df_vac)
    print(f'Loaded {len(df_vac)} steps from {_diag.name}')
    display(df_vac.tail())
else:
    print('No diagnostic file yet — run the simulation first.')
    df_vac = None


## 4. Plots

K_eff/K0 should remain ≈ 1.0 throughout (no compensation).


In [ ]:
if df_vac is not None:
    plot_neutralization_panel(df_vac, PLOTS_DIR, case_name='vacuum_reference')
    plot_keff_over_k0(df_vac, PLOTS_DIR, case_name='vacuum_reference')
    plt.show()
    print('Final K_eff/K0 =', df_vac['keff_over_k0'].iloc[-1])


## Physics checks (AGENTS.md)

- [ ] Beam velocity consistent with 30 keV proton energy
- [ ] Macroparticle weights are physically meaningful
- [ ] Species ordering in ParticleNumber is correct
- [ ] Ne and Ni increase for the correct physics reason
- [ ] K_eff/K0 never negative unless labelled overcompensation
- [ ] Beam envelope changes consistent with sign and magnitude of compensation
- [ ] Gas pressure and interaction length acceptable
- [ ] K_eff/K0 ≈ 1.0 (vacuum: no compensation)
